In [1]:
# ============================================
# HMOG DATASET - FULL OPTIMIZED CODE
# Target: >90% Accuracy for Continuous Authentication
# ============================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from scipy import signal
from scipy.fft import fft, fftfreq
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Conv1D, MaxPooling1D, Flatten, 
    Input, Concatenate, Bidirectional, BatchNormalization,
    GlobalAveragePooling1D, GlobalMaxPooling1D, Attention,
    Add, Multiply, Reshape, Permute, Lambda
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
    CSVLogger, TensorBoard
)
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

In [3]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("="*80)
print("🚀 HMOG DATASET - OPTIMIZED CONTINUOUS AUTHENTICATION")
print("Target: >90% Accuracy with Deep Learning")
print("="*80)

🚀 HMOG DATASET - OPTIMIZED CONTINUOUS AUTHENTICATION
Target: >90% Accuracy with Deep Learning


In [7]:
# ============================================
# 1. CONFIGURATION PARAMETERS
# ============================================

class Config:
    # Data parameters
    SEQUENCE_LENGTH = 256          # Window size
    STRIDE = 128                   # 50% overlap
    BATCH_SIZE = 32
    EPOCHS = 100
    MIN_SAMPLES_PER_SESSION = 500  # Minimum samples needed
    
    # Sensor configuration
    SENSORS = ['Accelerometer', 'Gyroscope', 'Magnetometer']
    
    # Model parameters
    TEST_SIZE = 0.2
    VALIDATION_SPLIT = 0.2
    RANDOM_STATE = 42
    
    # Path - SESUAIKAN DENGAN PATH ANDA
    BASE_PATH = r"C:\Users\natha\OneDrive\Documents\GitHub\Behavioral-Based_Authentication\public_dataset"
    
    # Deep Learning parameters
    LEARNING_RATE = 0.0005
    DROPOUT_RATE = 0.3
    L2_REG = 0.001

config = Config()
print(f"\n⚙️ Configuration:")
print(f"   SEQUENCE_LENGTH: {config.SEQUENCE_LENGTH}")
print(f"   STRIDE: {config.STRIDE}")
print(f"   BATCH_SIZE: {config.BATCH_SIZE}")



⚙️ Configuration:
   SEQUENCE_LENGTH: 256
   STRIDE: 128
   BATCH_SIZE: 32


In [8]:
# ============================================
# 2. DATA LOADING FUNCTIONS (FIXED)
# ============================================

print("\n" + "="*80)
print("📂 DATA LOADING")
print("="*80)

def load_sensor_data(session_path, sensor_name):
    """Load raw sensor data from CSV file"""
    file_path = os.path.join(session_path, f"{sensor_name}.csv")
    
    if not os.path.exists(file_path):
        return None
    
    try:
        df = pd.read_csv(file_path)
        
        # Look for x, y, z columns (case insensitive)
        data = []
        for axis in ['x', 'y', 'z']:
            found = False
            for col in df.columns:
                if col.lower() == axis:
                    data.append(df[col].values.astype(np.float32))
                    found = True
                    break
            if not found:
                # Try with column names like acc_x, gyro_x, etc.
                for col in df.columns:
                    if col.lower().endswith(f'_{axis}') or col.lower().endswith(axis):
                        data.append(df[col].values.astype(np.float32))
                        found = True
                        break
            if not found:
                data.append(np.zeros(len(df), dtype=np.float32))
        
        return np.column_stack(data)
    
    except Exception as e:
        return None

def interpolate_to_match(reference, target, target_length):
    """Interpolate array to match target length"""
    if len(target) == target_length:
        return target
    
    x_old = np.linspace(0, 1, len(target))
    x_new = np.linspace(0, 1, target_length)
    
    interpolated = np.zeros((target_length, target.shape[1]))
    for i in range(target.shape[1]):
        interpolated[:, i] = np.interp(x_new, x_old, target[:, i])
    
    return interpolated

def load_session_data(session_path):
    """Load all sensor data from a session with length alignment"""
    sensor_data_list = []
    
    for sensor in config.SENSORS:
        data = load_sensor_data(session_path, sensor)
        if data is not None:
            sensor_data_list.append(data)
    
    if len(sensor_data_list) < 2:  # Need at least 2 sensors
        return None
    
    # Find minimum length across all sensors
    min_length = min(len(data) for data in sensor_data_list)
    
    # Truncate all sensors to minimum length
    aligned_data = []
    for data in sensor_data_list:
        aligned_data.append(data[:min_length])
    
    # Concatenate all sensors horizontally
    combined_data = np.concatenate(aligned_data, axis=1)
    
    # Check if we have enough data
    if len(combined_data) < config.MIN_SAMPLES_PER_SESSION:
        return None
    
    return combined_data

def extract_windows(data, seq_length, stride):
    """Extract sliding windows from raw data"""
    if data is None or len(data) < seq_length:
        return []
    
    windows = []
    for start in range(0, len(data) - seq_length + 1, stride):
        window = data[start:start + seq_length].copy()
        windows.append(window)
    
    return windows

def advanced_preprocessing(window):
    """Advanced preprocessing: detrend, filter, normalize"""
    
    # Remove mean (DC offset)
    window = window - np.mean(window, axis=0)
    
    # Detrend (remove linear trend)
    x = np.arange(window.shape[0])
    for i in range(window.shape[1]):
        coeffs = np.polyfit(x, window[:, i], 1)
        trend = np.polyval(coeffs, x)
        window[:, i] = window[:, i] - trend
    
    # Apply Butterworth low-pass filter
    try:
        b, a = signal.butter(4, 0.2, 'low')
        window = signal.filtfilt(b, a, window, axis=0)
    except:
        pass
    
    # Z-score normalization per window
    mean = np.mean(window, axis=0)
    std = np.std(window, axis=0)
    std[std == 0] = 1
    window = (window - mean) / std
    
    return window.astype(np.float32)

def augment_data(window):
    """Simple data augmentation"""
    augmented = [window]
    
    # Add Gaussian noise
    noise = np.random.normal(0, 0.05, window.shape).astype(np.float32)
    augmented.append(window + noise)
    
    # Magnitude scaling
    scale = np.random.uniform(0.9, 1.1)
    augmented.append((window * scale).astype(np.float32))
    
    return augmented[:2]  # Return original + 1 augmented


📂 DATA LOADING


In [9]:
# ============================================
# 3. LOAD ALL DATA
# ============================================

print("\n📂 Loading HMOG dataset...")

# Get user folders
if not os.path.exists(config.BASE_PATH):
    print(f"\n❌ ERROR: Path tidak ditemukan: {config.BASE_PATH}")
    print("\n📁 Silakan ganti config.BASE_PATH dengan lokasi dataset Anda")
    exit()

user_folders = [f for f in os.listdir(config.BASE_PATH) 
               if os.path.isdir(os.path.join(config.BASE_PATH, f))]

print(f"\n📊 Found {len(user_folders)} users: {user_folders}")

all_sequences = []
all_labels = []
user_mapping = {}

for user_idx, user_folder in enumerate(user_folders):
    user_path = os.path.join(config.BASE_PATH, user_folder)
    user_mapping[user_idx] = user_folder
    
    # Find all session folders
    sessions = [f for f in os.listdir(user_path) 
               if os.path.isdir(os.path.join(user_path, f)) 
               and 'session' in f.lower()]
    sessions.sort()
    
    print(f"\n📂 Loading {user_folder}: {len(sessions)} sessions")
    
    user_sequences = []
    successful_sessions = 0
    
    for session in sessions:
        session_path = os.path.join(user_path, session)
        
        # Load sensor data
        sensor_data = load_session_data(session_path)
        
        if sensor_data is None:
            continue
        
        # Extract windows
        windows = extract_windows(sensor_data, config.SEQUENCE_LENGTH, config.STRIDE)
        
        if len(windows) == 0:
            continue
        
        successful_sessions += 1
        
        for window in windows:
            # Preprocess
            window_processed = advanced_preprocessing(window)
            
            # Add to list
            user_sequences.append(window_processed)
            all_sequences.append(window_processed)
            all_labels.append(user_idx)
            
            # Data augmentation (optional)
            if len(windows) < 50:  # Only augment if few windows
                aug_windows = augment_data(window_processed)
                for aug_window in aug_windows[1:]:
                    all_sequences.append(aug_window)
                    all_labels.append(user_idx)
                    user_sequences.append(aug_window)
    
    print(f"   ✅ {successful_sessions}/{len(sessions)} sessions -> {len(user_sequences)} sequences")

# Convert to numpy arrays
if len(all_sequences) == 0:
    print("\n❌ ERROR: No data loaded! Please check your dataset.")
    exit()

X = np.array(all_sequences, dtype=np.float32)
y = np.array(all_labels)

print(f"\n" + "="*80)
print("📊 DATA LOADING COMPLETE")
print("="*80)
print(f"✅ Total sequences: {len(X):,}")
print(f"✅ Sequence shape: {X.shape}")
print(f"✅ Total users: {len(np.unique(y))}")

# Display per-user distribution
print(f"\n📊 Data distribution per user:")
for user_idx in np.unique(y):
    count = np.sum(y == user_idx)
    print(f"   User {user_mapping[user_idx]}: {count:,} sequences")


📂 Loading HMOG dataset...

📊 Found 9 users: ['100669', '151985', '171538', '180679', '186676', '201848', '207969', '218719', '219303']

📂 Loading 100669: 24 sessions
   ✅ 24/24 sessions -> 11784 sequences

📂 Loading 151985: 24 sessions
   ✅ 24/24 sessions -> 10362 sequences

📂 Loading 171538: 24 sessions
   ✅ 24/24 sessions -> 9376 sequences

📂 Loading 180679: 24 sessions
   ✅ 24/24 sessions -> 7043 sequences

📂 Loading 186676: 24 sessions
   ✅ 24/24 sessions -> 10376 sequences

📂 Loading 201848: 24 sessions
   ✅ 24/24 sessions -> 13714 sequences

📂 Loading 207969: 24 sessions
   ✅ 24/24 sessions -> 11239 sequences

📂 Loading 218719: 24 sessions
   ✅ 24/24 sessions -> 15002 sequences

📂 Loading 219303: 24 sessions
   ✅ 24/24 sessions -> 4603 sequences

📊 DATA LOADING COMPLETE
✅ Total sequences: 93,499
✅ Sequence shape: (93499, 256, 9)
✅ Total users: 9

📊 Data distribution per user:
   User 100669: 11,784 sequences
   User 151985: 10,362 sequences
   User 171538: 9,376 sequences
   Use

In [10]:
# ============================================
# 4. TRAIN-TEST SPLIT
# ============================================

print("\n" + "="*80)
print("📊 TRAIN-TEST SPLIT")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, 
    random_state=config.RANDOM_STATE, 
    stratify=y
)

print(f"✅ Training samples: {X_train.shape[0]:,}")
print(f"✅ Testing samples: {X_test.shape[0]:,}")
print(f"✅ Input shape: (seq_len={config.SEQUENCE_LENGTH}, features={X.shape[2]})")
print(f"✅ Number of classes: {len(np.unique(y))}")


📊 TRAIN-TEST SPLIT
✅ Training samples: 74,799
✅ Testing samples: 18,700
✅ Input shape: (seq_len=256, features=9)
✅ Number of classes: 9


In [11]:

# ============================================
# 5. BUILD DEEP LEARNING MODELS
# ============================================

print("\n" + "="*80)
print("🏗️ BUILDING DEEP LEARNING MODELS")
print("="*80)

input_shape = (config.SEQUENCE_LENGTH, X.shape[2])
num_classes = len(np.unique(y))

def create_bilstm_model():
    """Bidirectional LSTM Model"""
    model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True, 
                          kernel_regularizer=l1_l2(l2=config.L2_REG)),
                     input_shape=input_shape),
        BatchNormalization(),
        Dropout(config.DROPOUT_RATE),
        
        Bidirectional(LSTM(64, return_sequences=True,
                          kernel_regularizer=l1_l2(l2=config.L2_REG))),
        BatchNormalization(),
        Dropout(config.DROPOUT_RATE),
        
        Bidirectional(LSTM(32, return_sequences=False,
                          kernel_regularizer=l1_l2(l2=config.L2_REG))),
        BatchNormalization(),
        Dropout(config.DROPOUT_RATE),
        
        Dense(128, activation='relu', kernel_regularizer=l1_l2(l2=config.L2_REG)),
        Dropout(config.DROPOUT_RATE),
        Dense(64, activation='relu'),
        Dropout(config.DROPOUT_RATE),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=config.LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

def create_cnn_lstm_model():
    """CNN-LSTM Hybrid Model"""
    model = Sequential([
        Conv1D(64, 5, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.25),
        
        Conv1D(128, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.25),
        
        Conv1D(256, 3, activation='relu', padding='same'),
        BatchNormalization(),
        GlobalAveragePooling1D(),
        
        Reshape((1, 256)),
        
        LSTM(64, return_sequences=False, kernel_regularizer=l1_l2(l2=config.L2_REG)),
        BatchNormalization(),
        Dropout(config.DROPOUT_RATE),
        
        Dense(128, activation='relu', kernel_regularizer=l1_l2(l2=config.L2_REG)),
        Dropout(config.DROPOUT_RATE),
        Dense(64, activation='relu'),
        Dropout(config.DROPOUT_RATE),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=config.LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Create models
models_dict = {
    'BiLSTM': create_bilstm_model(),
    'CNN-LSTM': create_cnn_lstm_model()
}

for name, model in models_dict.items():
    print(f"\n✅ {name}: {model.count_params():,} parameters")

# Callbacks
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6, verbose=0)
]


🏗️ BUILDING DEEP LEARNING MODELS

✅ BiLSTM: 365,833 parameters

✅ CNN-LSTM: 227,593 parameters


In [ ]:
# ============================================
# 6. TRAIN MODELS
# ============================================

print("\n" + "="*80)
print("🎯 TRAINING DEEP LEARNING MODELS")
print("="*80)

histories = {}

for name, model in models_dict.items():
    print(f"\n🔄 Training {name}...")
    
    history = model.fit(
        X_train, y_train,
        validation_split=config.VALIDATION_SPLIT,
        epochs=config.EPOCHS,
        batch_size=config.BATCH_SIZE,
        callbacks=callbacks,
        verbose=0
    )
    
    histories[name] = history
    
    train_acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    print(f"   ✅ Train Accuracy: {train_acc:.4f}")
    print(f"   ✅ Validation Accuracy: {val_acc:.4f}")



🎯 TRAINING DEEP LEARNING MODELS

🔄 Training BiLSTM...
